# Domain Classification Experiments

Load extracted domain features from `domain_analyzer/extractor/output`, build a labeled dataset, and evaluate multiple classifiers.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

data_dir = Path.cwd() / "extractor" / "output"
benign_path = data_dir / "benign_umbrella_features.csv"
phish_path = data_dir / "phishing_features.csv"

if not benign_path.exists() or not phish_path.exists():
    raise FileNotFoundError(f"Dataset files not found in {data_dir}")

df_benign = pd.read_csv(benign_path)
df_phish = pd.read_csv(phish_path)

print("benign shape:", df_benign.shape)
print("phish shape:", df_phish.shape)

# Align columns if needed
common_cols = df_benign.columns.intersection(df_phish.columns)
if len(common_cols) != len(df_benign.columns) or len(common_cols) != len(df_phish.columns):
    missing_benign = sorted(set(df_phish.columns) - set(df_benign.columns))
    missing_phish = sorted(set(df_benign.columns) - set(df_phish.columns))
    print("Columns missing in benign:", missing_benign[:10])
    print("Columns missing in phish:", missing_phish[:10])

df_benign = df_benign[common_cols]
df_phish = df_phish[common_cols]

df = pd.concat([df_benign, df_phish], ignore_index=True)
print("merged shape:", df.shape)
display(df.head(3))

benign shape: (50000, 139)
phish shape: (50000, 139)
merged shape: (100000, 139)


,domain_name,dns_has_dnskey,dns_dnssec_score,dns_zone_level,dns_zone_digit_count,dns_zone_len,dns_zone_entropy,dns_resolved_record_types,dns_ttl_avg,dns_ttl_stdev,...,rdap_ip_v6_count,rdap_ip_shortest_v4_prefix_len,rdap_ip_longest_v4_prefix_len,rdap_ip_shortest_v6_prefix_len,rdap_ip_longest_v6_prefix_len,rdap_ip_avg_admin_name_len,rdap_ip_avg_admin_name_entropy,rdap_ip_avg_admin_email_len,rdap_ip_avg_admin_email_entropy,class
0,hme-live-nitro-feedback-service.hmecloud.com,0,0.0,0,0,12,0.257080,1,450.0,1190.588090,...,0,14,14,0,0,0.0,0.000000,0.0,0.000000,benign
1,www.4digitalsignage.com,0,0.0,0,1,19,0.191692,1,450.0,1190.588090,...,0,11,11,0,0,13.0,0.249146,23.0,0.149881,benign
2,www.douyin.com.bytedns1.com,0,0.0,0,1,12,0.298747,1,7.5,19.843135,...,0,18,18,0,0,0.0,0.000000,0.0,0.000000,benign


In [2]:
# Prepare features and labels
df = df.copy()
df["class"] = df["class"].astype(str).str.lower()

label_map = {
    "benign": 0,
    "phish": 1
}
y = df["class"].map(label_map)

if y.isna().any():
    y = pd.to_numeric(df["class"], errors="coerce")

valid_mask = y.isin([0, 1])
df = df.loc[valid_mask].copy()
y = y.loc[valid_mask].astype(int)

drop_cols = ["class"]
if "domain_name" in df.columns:
    drop_cols.append("domain_name")

X = df.drop(columns=drop_cols)
X = X.replace({True: 1, False: 0})
X = X.apply(pd.to_numeric, errors="coerce")

print("X shape:", X.shape)

X shape: (100000, 137)


In [3]:
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
 )
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

def make_pipeline(model, scale=True):
    steps = [("imputer", SimpleImputer(strategy="median"))]
    if scale:
        steps.append(("scaler", StandardScaler()))
    steps.append(("model", model))
    return Pipeline(steps)

models = {
    "LogisticRegression": make_pipeline(
        LogisticRegression(max_iter=500, class_weight="balanced")
    ),
    "DecisionTree": make_pipeline(
        DecisionTreeClassifier(class_weight="balanced", random_state=42),
        scale=False,
    ),
    "RandomForest": make_pipeline(
        RandomForestClassifier(
            n_estimators=300,
            random_state=42,
            class_weight="balanced",
            n_jobs=-1,
        ),
        scale=False,
    ),
    "AdaBoost": make_pipeline(
        AdaBoostClassifier(random_state=42),
        scale=False,
    ),
}


models["XGBoost"] = make_pipeline(
    XGBClassifier(
        n_estimators=300,
        learning_rate=0.1,
        max_depth=6,
        subsample=0.9,
        colsample_bytree=0.9,
        random_state=42,
        eval_metric="logloss",
        tree_method="hist",
    ),
    scale=False,
)



models["LightGBM"] = make_pipeline(
    LGBMClassifier(
        n_estimators=300,
        learning_rate=0.05,
        num_leaves=31,
        random_state=42,
    ),
    scale=False,
)


def evaluate_model(name, model, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    metrics = {
        "model": name,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1": f1_score(y_test, y_pred, zero_division=0),
    }

    y_score = None
    if hasattr(model, "predict_proba"):
        y_score = model.predict_proba(X_test)[:, 1]
    elif hasattr(model, "decision_function"):
        y_score = model.decision_function(X_test)

    if y_score is not None:
        metrics["roc_auc"] = roc_auc_score(y_test, y_score)
        metrics["avg_precision"] = average_precision_score(y_test, y_score)
    else:
        metrics["roc_auc"] = np.nan
        metrics["avg_precision"] = np.nan

    return metrics, y_pred

results = []
predictions = {}

for name, model in models.items():
    print(f"Training {name}...")
    metrics, y_pred = evaluate_model(name, model, X_train, X_test, y_train, y_test)
    results.append(metrics)
    predictions[name] = y_pred

results_df = (
    pd.DataFrame(results)
    .set_index("model")
    .sort_values(["f1", "roc_auc"], ascending=False)
 )

display(results_df)

Evaluating LogisticRegression...
Evaluating DecisionTree...
Evaluating RandomForest...
Evaluating AdaBoost...
Evaluating XGBoost...
Evaluating LightGBM...
[LightGBM] [Info] Number of positive: 40000, number of negative: 40000
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.022806 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 13151
[LightGBM] [Info] Number of data points in the train set: 80000, number of used features: 134
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000


/home/ubuntu/Desktop/Phishing-Detection-Engine/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/ubuntu/Desktop/Phishing-Detection-Engine/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


,accuracy,precision,recall,f1,roc_auc,avg_precision
model,,,,,,
XGBoost,0.99020,0.993060,0.9873,0.990171,0.999383,0.999422
LightGBM,0.98910,0.992052,0.9861,0.989067,0.999310,0.999354
RandomForest,0.98700,0.994617,0.9793,0.986899,0.999080,0.999145
DecisionTree,0.96840,0.966348,0.9706,0.968469,0.968400,0.952637
LogisticRegression,0.94080,0.944176,0.9370,0.940574,0.985097,0.985646
AdaBoost,0.93685,0.953211,0.9188,0.935689,0.985001,0.985988


In [ ]:
from pathlib import Path
import joblib
from sklearn.metrics import confusion_matrix, classification_report

best_model_name = results_df.index[0]
best_model = models[best_model_name]

best_model.fit(X_train, y_train)
best_pred = best_model.predict(X_test)

print("Best model:", best_model_name)
print(classification_report(y_test, best_pred, target_names=["benign", "phish"]))

cm = confusion_matrix(y_test, best_pred)
cm_df = pd.DataFrame(
    cm,
    index=["true_benign", "true_phish"],
    columns=["pred_benign", "pred_phish"],
)

display(cm_df)

best_model.fit(X, y)

model_dir = Path.cwd() / "models"
model_dir.mkdir(parents=True, exist_ok=True)
model_path = model_dir / f"best_model_{best_model_name}_full.joblib"
joblib.dump(best_model, model_path)
print(f"Saved full-data model to {model_path}")

Best model: XGBoost
              precision    recall  f1-score   support

      benign       0.99      0.99      0.99     10000
       phish       0.99      0.99      0.99     10000

    accuracy                           0.99     20000
   macro avg       0.99      0.99      0.99     20000
weighted avg       0.99      0.99      0.99     20000



,pred_benign,pred_phish
true_benign,9931,69
true_phish,127,9873


Saved best model to /home/ubuntu/Desktop/Phishing-Detection-Engine/domain_analyzer/models/best_model_XGBoost.joblib
